In [10]:
from datasets import load_dataset
from langchain_community.llms import Ollama

# Load dataset evaluasi
eval_dataset = load_dataset("json", data_files="/Users/novay/Applications/Project/Enterwind/lexindo/python/lexindo-llm/03_finetune_dataset/data/split/ragas_eval.jsonl")["train"]
eval_dataset[0]

{'question': 'Apa tujuan Manajemen Sumber Daya Manusia menurut Pasal 20?',
 'ground_truth': 'Berdasarkan Peraturan Bupati Nomor 45 Tahun 2022 tentang Sistem Pemerintahan Berbasis Elektronik Dalam Penyelenggaraan Pemerintahan Daerah, Pasal 20 menyatakan bahwa:\n\nPasal 20 Manajemen Sumber Daya Manusia sebagaimana dimaksud dalam Pasal 15 ayat (2) huruf e dilakukan untuk menjamin ketersediaan dan meningkatkan kompetensi pegawai ASN dalam penyelenggaraan SPBE melalui pendidikan dan pelatihan.\n\n(Sumber: Peraturan Bupati Nomor 45 Tahun 2022, Pasal 20)',
 'answer': 'Berdasarkan Peraturan Bupati Nomor 45 Tahun 2022 tentang Sistem Pemerintahan Berbasis Elektronik Dalam Penyelenggaraan Pemerintahan Daerah, Pasal 20 menyatakan bahwa:\n\nPasal 20 Manajemen Sumber Daya Manusia sebagaimana dimaksud dalam Pasal 15 ayat (2) huruf e dilakukan untuk menjamin ketersediaan dan meningkatkan kompetensi pegawai ASN dalam penyelenggaraan SPBE melalui pendidikan dan pelatihan.',
 'contexts': ['(Sumber:Peratu

In [24]:
# LLM via Ollama
llm_lexindo = Ollama(model="lexindo-custom", temperature=0.3, num_ctx=512)

In [46]:
import json
import re
from tqdm import tqdm

def clean_lexindo_text(text):
    if not text: return ""
    
    # 1. Hapus label "Sumber:" (Sangat aman)
    text = re.sub(r'\(?Sumber\s?:.*?\)?', '', text, flags=re.IGNORECASE)
    
    # 2. Hapus referensi di akhir jika diawali kurung buka dan ada di ujung string
    # Kita cari kurung tutup di paling akhir, lalu tarik ke belakang sampai ketemu '(' 
    # tapi hanya jika di dalamnya ada kata kunci hukum.
    text = re.sub(r'\((?:[^()]*?(?:Peraturan|Perbup|Nomor|Pasal|Tahun)[^()]*?)\)\.?$', '', text, flags=re.IGNORECASE).strip()
    
    # 3. Hard-fix untuk kasus Anda: Kurung tutup yang tertinggal tanpa kurung buka
    # Kita cari pola Peraturan/Pasal yang muncul SETELAH titik terakhir atau di 60 karakter terakhir
    if ')' in text[-10:] and ('Pasal' in text[-60:] or 'Peraturan' in text[-100:]):
        # Cari posisi awal dari pola referensi di bagian ujung saja
        parts = re.split(r'(\.?\s?\(?Peraturan Bupati|Perbup)', text)
        if len(parts) > 1:
            # Ambil semua bagian kecuali bagian terakhir yang berisi referensi
            text = "".join(parts[:-1])
            
    return text.strip().rstrip(').')

def batch_generate_safe(dataset, model, output_file="hasil_eval_incremental.jsonl"):
    print(f"Memulai proses generate untuk {len(dataset)} pertanyaan...")
    
    with open(output_file, "a", encoding="utf-8") as f:
        for i, data in enumerate(tqdm(dataset)):
            try:
                question = data["question"]
                
                # Memanggil model (Gunakan invoke untuk Ollama atau generate untuk Unsloth)
                raw_response = model.invoke(f"Pertanyaan: {question}\nJawaban akurat:").strip()
                
                # Bersihkan kedua belah pihak
                clean_answer = clean_lexindo_text(raw_response)
                clean_ground_truth = clean_lexindo_text(data.get("ground_truth", ""))
                
                entry = {
                    "id": i,
                    "question": question,
                    "answer": clean_answer,
                    "ground_truth": clean_ground_truth
                }

                # Tampilkan untuk verifikasi
                if i % 10 == 0: # Print setiap 10 agar tidak memenuhi layar
                    print(f"\n[Check ID {i}] Answer: {entry['answer']}")

                f.write(json.dumps(entry, ensure_ascii=False) + "\n")
                f.flush()
                
            except Exception as e:
                print(f"\nError pada indeks {i}: {e}")
                continue

In [47]:
# Eksekusi
batch_generate_safe(eval_dataset, llm_lexindo)

Memulai proses generate untuk 1119 pertanyaan...


  0%|          | 1/1119 [00:01<34:23,  1.85s/it]


[Check ID 0] Answer: Tujuannya adalah meningkatkan kualitas SDM secara optimal melalui pengelolaan SDM yang baik.

 Peraturan Bupati


  1%|          | 11/1119 [03:07<2:56:57,  9.58s/it]


[Check ID 10] Answer: Berdasarkan Pasal 15 ayat (2) Peraturan Bupati Nomor 59 Tahun 2021, Koordinator Wilayah Kecamatan dipimpin oleh Kepala Desa. Untuk menjadi koordinator wilayah kecamatan, desa harus memenuhi persyaratan: a. memiliki pengalaman di bidang pemerintahan dan/atau pengelolaan; b. memahami teknologi informasi dan komunikasi; c. mampu mengelola perangkat desa; d. mampu membina dan mengembangkan pembudayaan kepedulian terhadap perempuan; e. mampu mengelola anggaran; f. memahami prinsip-prinsip penyelenggaraan Pemerintahan Desa; g. memiliki keterampilan di bidang teknologi informasi dan komunikasi; h. memiliki kecerdasan dan kreativitas tingkat menengah; i. berusia paling rendah 30 tahun; j. memahami prinsip-prinsip penyelenggaraan Pemerintahan Desa; k. mampu mengelola perangkat desa; l. mampu membina dan mengembangkan pembudayaan kepedulian terhadap perempuan; m. mampu mengelola anggaran; n. memahami teknologi informasi dan komunikasi; o. memiliki pengalaman di bidang peme

...

 98%|█████████▊| 1101/1119 [3:53:45<05:44, 19.16s/it]


[Check ID 1100] Answer: Berdasarkan Pasal 1 angka 7 Peraturan Bupati Nomor 61 Tahun 2020, Hibah adalah pemberian barang/jasa/uang yang tidak mengikat dan tidak memaksa oleh penerima hibah.  Peraturan Bupati


 99%|█████████▉| 1111/1119 [3:56:14<01:34, 11.86s/it]


[Check ID 1110] Answer: Berdasarkan Pasal 4 ayat (2) Peraturan Bupati Nomor 32 Tahun 2023, Penetapan dan Penegasan Batas Desa dilakukan secara administrasi tidak mengubah hak atas tanah, yang berarti pemilik tanah tetap mempunyai kewenangan, hak, kewajiban, dan hak-hak lainnya sesuai dengan peraturan perundang-undangan.

 Peraturan Bupati


100%|██████████| 1119/1119 [3:57:28<00:00, 12.73s/it]
